# Proyecto Olist — Fundamentos Probabilísticos de ML
### Analista de experiencia del cliente en un marketplace

Este notebook aplica los 11 conceptos probabilísticos del curso al dataset 
de Olist (Brazilian E-Commerce), respondiendo preguntas concretas de negocio 
sobre satisfacción del cliente y comportamiento de vendedores.

## 0. Imports y carga de datos

In [3]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

pd.set_option("display.width", 100)
pd.set_option('display.max_columns', None)
np.set_printoptions(precision=4, suppress=True)

In [4]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/product_category_name_translation.csv')

print(f"orders: {orders.shape}, items: {items.shape}, payments: {payments.shape}")
print(f"reviews: {reviews.shape}, products: {products.shape}")

orders: (99441, 8), items: (112650, 7), payments: (103886, 5)
reviews: (99224, 7), products: (32951, 9)


## 0.1 Limpieza y construcción del dataframe maestro

**Por qué un dataframe maestro:** Olist distribuye la información en 9 
archivos CSV relacionados por `order_id` (y por `product_id`/`seller_id` 
en cascada). Sin embargo, la mayoría de las preguntas de negocio de este 
análisis requieren cruzar información de al menos dos tablas — por 
ejemplo, "¿la entrega tardía se asocia con reseñas negativas?" necesita 
`retraso_dias` (calculado desde `orders`) junto con `review_score` 
(de `reviews`). Construir un único dataframe maestro, con una fila por 
pedido y todas las columnas relevantes ya unidas, evita repetir joins 
parciales e inconsistentes en cada uno de los 11 bloques de análisis, y 
deja una sola fuente de verdad verificable.


In [5]:
# ============================================================
# PASO 1: Convertir las columnas de fecha (llegan como texto/str) a datetime
# ============================================================ 
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# ============================================================
# PASO 2: Filtrar a pedidos completados y bien documentados
# ============================================================
# Excluimos:
# (a) pedidos que NO llegaron a estado 'delivered' (cancelados, en tránsito,
#     etc.) -- no tiene sentido medir "tiempo de entrega" de algo que nunca
#     se entregó.
# (b) los 8 casos inconsistentes detectados en la exploración: marcados como
#     'delivered' pero SIN fecha de entrega registrada (probable error de
#     captura de datos de Olist).
orders_delivered = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notnull())
].copy()  # .copy() evita el warning de pandas al modificar un subconjunto filtrado

# ============================================================
# PASO 3: Verificar cuántos registros quedaron y cuántos se excluyeron
# ============================================================
print(f"Pedidos originales: {len(orders):,}")
print(f"Pedidos delivered con fecha completa: {len(orders_delivered):,}")
print(f"Excluidos: {len(orders) - len(orders_delivered):,}")

Pedidos originales: 99,441
Pedidos delivered con fecha completa: 96,470
Excluidos: 2,971


**Decisiones de limpieza aplicadas:**
- Convertimos las columnas de fecha (texto) a tipo `datetime`, necesario 
  para calcular diferencias de tiempo.
- Filtramos a pedidos con `order_status == 'delivered'` y con 
  `order_delivered_customer_date` no nulo, excluyendo pedidos cancelados, 
  en tránsito, o con inconsistencias de registro (8 casos marcados como 
  entregados sin fecha de entrega) — el análisis de tiempos de entrega 
  solo tiene sentido sobre pedidos completados y bien documentados.
- Derivamos `tiempo_entrega_dias` (duración real del envío) y 
  `retraso_dias` (diferencia contra la fecha estimada; positivo = tardío).